
# Phase 10B — Non-Degenerate Spectral Analysis

This is the follow-up to the corrected Phase 10.

It fixes one important reporting issue:

If a task has zero total spectral energy, assigning its normalized energy fraction to zero and then averaging across all tasks produces a misleading result.

Example:

- 3 tasks have all their energy at order 0;
- 9 tasks are identically zero.

A naive average gives

\[
\text{mean order-0 fraction}=0.25,
\]

even though **every nonzero task has 100% of its energy at order 0**.

This notebook reports separately:

\[
P(E_{\text{tot}}>0)
\]

and

\[
E\!\left[\frac{E_k}{E_{\text{tot}}}\mid E_{\text{tot}}>0\right].
\]


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import comb



## 1. Load a subset surface

Expected columns:

- `provider`
- `task_id`
- optional `family`
- `mask`
- signal column, e.g. `F`, `p_correct`, `mean_logprob_gold`

There must be 16 masks per complete task for \(d=4\).


In [ ]:

DATA_PATH = None
SIGNAL_COL = "F"
N_BITS = 4
N_STATES = 2**N_BITS

def load_table(path):
    p = Path(path)
    if p.suffix.lower() == ".csv":
        return pd.read_csv(p)
    if p.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(p)
    if p.suffix.lower() in {".jsonl", ".ndjson"}:
        return pd.read_json(p, lines=True)
    raise ValueError(p.suffix)

if DATA_PATH is None:
    surface = None
    print("Set DATA_PATH and SIGNAL_COL.")
else:
    surface = load_table(DATA_PATH)
    display(surface.head())


In [ ]:

def fwht(a):
    a = np.asarray(a, dtype=float).copy()
    h = 1
    n = len(a)
    while h < n:
        for i in range(0, n, 2*h):
            x = a[i:i+h].copy()
            y = a[i+h:i+2*h].copy()
            a[i:i+h] = x+y
            a[i+h:i+2*h] = x-y
        h *= 2
    return a

def popcount(x):
    return int(x).bit_count()

def walsh_energy(f, d=N_BITS):
    f = np.asarray(f,float)
    coeff = fwht(f) / len(f)
    rows = []
    for order in range(d+1):
        idx = [i for i in range(2**d) if popcount(i)==order]
        rows.append({
            "order": order,
            "energy": float(np.sum(coeff[idx]**2)),
        })
    return pd.DataFrame(rows)


In [ ]:

def task_spectral_table(surface, signal_col=SIGNAL_COL):
    base_cols = ["provider","task_id"]
    if "family" in surface.columns:
        base_cols.append("family")

    rows = []
    for keys, g in surface.groupby(base_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        meta = dict(zip(base_cols, keys))

        g = g.sort_values("mask")
        if g["mask"].nunique() != N_STATES:
            continue

        f = g[signal_col].to_numpy(float)
        e = walsh_energy(f, N_BITS)
        total = float(e["energy"].sum())
        nonzero = total > 1e-15

        for _, r in e.iterrows():
            rows.append({
                **meta,
                "order": int(r["order"]),
                "energy": float(r["energy"]),
                "total_energy": total,
                "nonzero_signal": nonzero,
                "energy_fraction": (
                    float(r["energy"]/total) if nonzero else np.nan
                ),
                "signal_std": float(np.std(f, ddof=0)),
                "signal_range": float(np.ptp(f)),
                "n_unique_signal": int(len(np.unique(np.round(f,12)))),
            })

    return pd.DataFrame(rows)

if surface is not None:
    task_spec = task_spectral_table(surface, SIGNAL_COL)
    display(task_spec.head(20))



## 2. Correct aggregation

We report:
- fraction of tasks with nonzero signal energy;
- absolute mean energy by order over all tasks;
- normalized energy fraction **only among nonzero tasks**.


In [ ]:

def aggregate_spectral(task_spec):
    if task_spec.empty:
        return pd.DataFrame(), pd.DataFrame()

    task_keys = ["provider","task_id"]
    task_nonzero = (
        task_spec[task_keys + ["nonzero_signal"]]
        .drop_duplicates()
    )

    coverage = (
        task_nonzero
        .groupby("provider")
        .agg(
            n_tasks=("task_id","nunique"),
            n_nonzero=("nonzero_signal","sum"),
            fraction_nonzero=("nonzero_signal","mean"),
        )
        .reset_index()
    )

    abs_summary = (
        task_spec
        .groupby(["provider","order"])
        .agg(
            mean_abs_energy=("energy","mean"),
            std_abs_energy=("energy","std"),
        )
        .reset_index()
    )

    nz = task_spec[task_spec["nonzero_signal"]].copy()
    if len(nz):
        frac_summary = (
            nz
            .groupby(["provider","order"])
            .agg(
                mean_energy_fraction_nonzero=("energy_fraction","mean"),
                std_energy_fraction_nonzero=("energy_fraction","std"),
                n_nonzero_tasks=("task_id","nunique"),
            )
            .reset_index()
        )
    else:
        frac_summary = pd.DataFrame()

    return coverage, abs_summary.merge(
        frac_summary, on=["provider","order"], how="left"
    )

if surface is not None:
    coverage, summary = aggregate_spectral(task_spec)
    display(coverage)
    display(summary)


In [ ]:

def plot_corrected_summary(summary):
    for provider, g in summary.groupby("provider"):
        plt.figure(figsize=(7,4))
        plt.plot(g["order"], g["mean_abs_energy"], marker="o")
        plt.xlabel("Walsh order")
        plt.ylabel("mean absolute spectral energy")
        plt.title(f"Absolute spectral energy — {provider}")
        plt.show()

        if "mean_energy_fraction_nonzero" in g.columns:
            plt.figure(figsize=(7,4))
            plt.plot(
                g["order"],
                g["mean_energy_fraction_nonzero"],
                marker="o"
            )
            plt.ylim(-0.02,1.02)
            plt.xlabel("Walsh order")
            plt.ylabel("mean fraction, conditioned on nonzero tasks")
            plt.title(f"Normalized spectral energy | nonzero — {provider}")
            plt.show()

# if surface is not None:
#     plot_corrected_summary(summary)



## 3. Laplacian and heat-flow only on informative tasks

A constant signal has

\[
f^\top Lf=0
\]

and cannot exhibit heat-flow smoothing.

This section automatically separates:
- flat/zero tasks;
- non-flat tasks worth plotting.


In [ ]:

def hypercube_laplacian(d):
    n = 2**d
    A = np.zeros((n,n), dtype=float)
    for x in range(n):
        for bit in range(d):
            A[x, x ^ (1<<bit)] = 1.0
    return np.diag(A.sum(axis=1)) - A

L = hypercube_laplacian(N_BITS)
eigvals, eigvecs = np.linalg.eigh(L)

expected = []
for k in range(N_BITS+1):
    expected.extend([2*k]*comb(N_BITS,k))
assert np.allclose(eigvals, expected)

def heat_flow(f, t):
    f = np.asarray(f,float)
    return eigvecs @ (
        np.exp(-t*eigvals) * (eigvecs.T @ f)
    )

def heat_roughness(f, t):
    ft = heat_flow(f,t)
    return float(ft @ L @ ft)

def nonflat_tasks(surface, signal_col=SIGNAL_COL, tol=1e-12):
    keys = ["provider","task_id"]
    if "family" in surface.columns:
        keys.append("family")
    rows = []
    for meta, g in surface.groupby(keys, dropna=False):
        g = g.sort_values("mask")
        if g["mask"].nunique() != N_STATES:
            continue
        f = g[signal_col].to_numpy(float)
        if np.std(f, ddof=0) > tol:
            rows.append((meta, g, f))
    return rows

if surface is not None:
    informative = nonflat_tasks(surface, SIGNAL_COL)
    print("informative non-flat tasks:", len(informative))

    for meta, g, f in informative[:5]:
        ts = np.linspace(0, 4, 33)
        rough = [heat_roughness(f,t) for t in ts]
        plt.figure(figsize=(6,3.5))
        plt.plot(ts, rough)
        plt.xlabel("t")
        plt.ylabel(r"$f_t^T L f_t$")
        plt.title(f"Heat-flow roughness — {meta}")
        plt.show()



## 4. Interpretation gate

A useful Phase-10B run should satisfy:

- a nontrivial fraction of tasks has `signal_std > 0`;
- normalized energy among those tasks is meaningful;
- heat-flow roughness decays from a nonzero initial value;
- different task families exhibit different order profiles.

If everything is still flat, the next intervention belongs upstream in task generation/scoring, not in spectral mathematics.
